# 🌿 Project 26 — Calculate EVI

Great. This project is important because you're moving from **basic vegetation indices** to understanding **why different indices exist**.

You already know NDVI. Now we'll calculate **EVI (Enhanced Vegetation Index)** and compare the two.

---

# 1. Introduce the Project

## Goal

Calculate **EVI** from Sentinel-2 imagery and compare it with **NDVI**.

NDVI is excellent for vegetation monitoring, but it can become less sensitive when vegetation is very dense and can be affected by atmospheric and soil-background effects.

EVI was designed to improve vegetation monitoring under some of these conditions.

### Real-world applications

EVI is useful for:

- 🌳 Dense forest monitoring
- 🌾 Crop monitoring
- 🌍 Ecosystem studies
- 🌱 Vegetation productivity
- 🌡 Climate and environmental studies

---

# 2. Big Picture

You've previously done:

```text
Sentinel-2 Image
       ↓
Red + NIR
       ↓
NDVI
       ↓
Vegetation Map
````

Today:

```text
Sentinel-2 Image
       ↓
 ┌─────┴─────┐
 ↓           ↓
NDVI        EVI
 ↓           ↓
 └─────┬─────┘
       ↓
   Compare
```

The important difference is that **EVI uses more spectral information than NDVI**.

---

# 3. NDVI vs EVI

## NDVI

The formula is:

```text
NDVI = (NIR - Red) / (NIR + Red)
```

For Sentinel-2:

```text
NDVI = (B8 - B4) / (B8 + B4)
```

---

## EVI

The commonly used formula is:

```text
EVI = G × (NIR - Red)
      -----------------------------
      NIR + C1×Red - C2×Blue + L
```

Common coefficients:

```text
G  = 2.5
C1 = 6
C2 = 7.5
L  = 1
```

For Sentinel-2:

```text
NIR  → B8
Red  → B4
Blue → B2
```

So:

```text
EVI = 2.5 × (B8 - B4)
      ------------------------
      B8 + 6×B4 - 7.5×B2 + 1
```

### Key idea

NDVI:

```text
Red + NIR
```

EVI:

```text
Blue + Red + NIR
```

That's why EVI can provide additional information related to atmospheric and background effects.

---

# 4. Important Concept — Atmospheric Correction

You already know that satellite sensors measure radiation **after it passes through the atmosphere**.

The atmosphere can affect the signal because of things like:

* Aerosols
* Scattering
* Absorption

Conceptually:

```text
Sun
 ↓
Atmosphere
 ↓
Vegetation
 ↓
Atmosphere
 ↓
Satellite
```

The satellite therefore doesn't measure vegetation in a perfectly isolated environment.

EVI includes a correction term using the **Blue band** to help account for some atmospheric influences.

### Important clarification

EVI is **not a replacement for atmospheric correction**.

It is an index designed to be more resistant to certain atmospheric and background effects.

Also, we're using the Sentinel-2 **Surface Reflectance** product, which has already undergone atmospheric processing. So don't think:

> "EVI itself performs atmospheric correction."

That's not correct.

---

# 5. Step 1 — Define the Study Area

We'll use the same Berlin study area.

```javascript
var studyArea = ee.Geometry.Polygon([
  [
    [13.20,52.35],
    [13.65,52.35],
    [13.65,52.65],
    [13.20,52.65],
    [13.20,52.35]
  ]
]);
```

---

# 6. Step 2 — Load Sentinel-2

```javascript
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate('2024-06-01','2024-06-30')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(studyArea);
```

### What's happening?

```text
ImageCollection
      ↓
filterBounds()
      ↓
filterDate()
      ↓
sort()
      ↓
first()
      ↓
ee.Image
```

You've used this workflow many times now.

---

# 7. Step 3 — Calculate NDVI

You already know this:

```javascript
var ndvi = image
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');
```

Remember:

```text
B8 = NIR
B4 = Red
```

---

# 8. Step 4 — Calculate EVI ⭐

Now we use the EVI formula directly.

```javascript
var evi = image.expression(
  '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
  {
    NIR: image.select('B8'),
    RED: image.select('B4'),
    BLUE: image.select('B2')
  }
).rename('EVI');
```

This introduces an important GEE method:

## `expression()`

`expression()` allows us to write mathematical formulas directly.

Instead of manually building:

```text
subtract()
multiply()
add()
divide()
```

we can write the formula in a way that looks closer to the mathematical equation.

---

# 9. Understand `expression()`

This:

```text
'NIR - RED'
```

doesn't mean the text is being subtracted.

We tell Earth Engine what `NIR` and `RED` represent here:

```javascript
{
  NIR: image.select('B8'),
  RED: image.select('B4'),
  BLUE: image.select('B2')
}
```

So Earth Engine interprets:

```text
NIR  → B8
RED  → B4
BLUE → B2
```

Then calculates the formula **pixel by pixel**.

---

# 10. Step 5 — Visualization

We'll use a similar visualization for both indices.

```javascript
var vegetationVis = {
  min: -1,
  max: 1,
  palette: [
    'brown',
    'yellow',
    'green'
  ]
};
```

---

# 11. Step 6 — Display NDVI and EVI

```javascript
Map.centerObject(studyArea, 10);

Map.addLayer(
  ndvi,
  vegetationVis,
  'NDVI'
);

Map.addLayer(
  evi,
  vegetationVis,
  'EVI'
);
```

Turn the layers on and off.

Look at areas with:

* Dense vegetation
* Sparse vegetation
* Urban areas
* Bare soil

Ask yourself:

> Where does EVI look different from NDVI?

---

# ⭐ Bonus — Calculate EVI − NDVI

We can directly compare the two indices.

```javascript
var difference = evi.subtract(ndvi);
```

Then display it:

```javascript
var differenceVis = {
  min: -0.5,
  max: 0.5,
  palette: [
    'red',
    'white',
    'blue'
  ]
};

Map.addLayer(
  difference,
  differenceVis,
  'EVI - NDVI'
);
```

Conceptually:

```text
EVI > NDVI
    ↓
Positive value

EVI < NDVI
    ↓
Negative value
```

This isn't necessarily saying one index is "correct" and the other is "wrong." They measure vegetation differently.

---

# Complete Code

```javascript
// 1. Study Area
var studyArea = ee.Geometry.Polygon([
  [
    [13.20,52.35],
    [13.65,52.35],
    [13.65,52.65],
    [13.20,52.65],
    [13.20,52.35]
  ]
]);

// 2. Sentinel-2 Image
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate('2024-06-01','2024-06-30')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(studyArea);

// 3. NDVI
var ndvi = image
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');

// 4. EVI
var evi = image.expression(
  '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
  {
    NIR: image.select('B8'),
    RED: image.select('B4'),
    BLUE: image.select('B2')
  }
).rename('EVI');

// 5. Visualization
var vegetationVis = {
  min: -1,
  max: 1,
  palette: [
    'brown',
    'yellow',
    'green'
  ]
};

// 6. Display
Map.centerObject(studyArea, 10);

Map.addLayer(
  ndvi,
  vegetationVis,
  'NDVI'
);

Map.addLayer(
  evi,
  vegetationVis,
  'EVI'
);

// 7. Bonus: Difference
var difference = evi.subtract(ndvi);

var differenceVis = {
  min: -0.5,
  max: 0.5,
  palette: [
    'red',
    'white',
    'blue'
  ]
};

Map.addLayer(
  difference,
  differenceVis,
  'EVI - NDVI'
);
```

---

# Things to Remember

## 1. NDVI

```text
NDVI = (NIR - Red)
       ------------
       (NIR + Red)
```

Sentinel-2:

```text
NIR → B8
Red → B4
```

---

## 2. EVI

```text
EVI = 2.5 × (NIR - Red)
      -------------------------
      NIR + 6Red - 7.5Blue + 1
```

Sentinel-2:

```text
NIR  → B8
Red  → B4
Blue → B2
```

---

## 3. Main Difference

| NDVI                                                 | EVI                                                    |
| ---------------------------------------------------- | ------------------------------------------------------ |
| Uses Red + NIR                                       | Uses Blue + Red + NIR                                  |
| Simpler                                              | More complex                                           |
| Can saturate in dense vegetation                     | Better sensitivity in some dense vegetation conditions |
| More affected by some atmospheric/background effects | Designed to reduce some of these effects               |

---

# Common Beginner Mistakes

## ❌ Mistake 1: Using the wrong Sentinel-2 bands

For this EVI formula:

```text
Blue → B2
Red  → B4
NIR  → B8
```

---

## ❌ Mistake 2: Thinking EVI is simply "better NDVI"

Not necessarily.

The choice depends on the research question, vegetation conditions, sensor, preprocessing, and study environment.

---

## ❌ Mistake 3: Thinking EVI performs all atmospheric correction

EVI is designed to reduce sensitivity to some atmospheric and canopy/background effects.

It does **not** replace proper atmospheric correction.

---

## ❌ Mistake 4: Forgetting the coefficients

EVI isn't:

```text
(NIR - Red) / (NIR + Red)
```

That's NDVI.

EVI contains:

```text
2.5
6
7.5
1
```

These coefficients are part of the standard EVI formulation.

---

# Your Practice — Project 26.1

Create an NDVI vs EVI comparison for a study area of your choice.

### Requirements

1. Define a study area.
2. Load a Sentinel-2 Surface Reflectance image.
3. Calculate NDVI.
4. Calculate EVI using `expression()`.
5. Display both indices.
6. Compare areas with sparse and dense vegetation.
7. **Bonus:** Calculate `EVI - NDVI`.

### Conceptual Questions

**Question 1:**
Why does EVI use the **Blue band**, while NDVI only uses Red and NIR?

**Question 2:**
What is the main reason EVI can be useful in **dense vegetation** where NDVI may become less sensitive?

**Question 3:**
What type of GEE object does this return?

```javascript
var evi = image.expression(...);
```

Is it an `ee.Image` or an `ee.ImageCollection`?

```
```
